In [ ]:
# ============================================================
# MODEL B: Conservative Separator
# Purpose: Under-connect rather than over-connect.
#          Prevents mergers in packed regions.
# Loss: CE + Dice + FocalLoss (NO medial surface recall)
# Augmentation: Stronger boundary/contrast augmentation
# ============================================================

import os, sys, gc, time, math, random, warnings
warnings.filterwarnings("default")  # don't globally silence important warnings

pipeline_start = time.time()
MAX_RUNTIME = 9 * 3600  # Kaggle GPU hard limit is 9h for notebooks

def elapsed_h():
    return (time.time() - pipeline_start) / 3600

def remaining_h():
    return (MAX_RUNTIME / 3600) - elapsed_h()

def check_budget(tag=""):
    e, r = elapsed_h(), remaining_h()
    print(f"[TIME] {tag}: {e:.2f}h elapsed, {r:.2f}h left")
    return e, r

check_budget("Pipeline start")

# Reproducibility (important for aug-heavy Model B)
SEED = 1337
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
try:
    import numpy as np
    np.random.seed(SEED)
except Exception:
    pass

try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
except Exception:
    pass

# TIFF IO: do NOT pip install in a Kaggle no-internet environment
try:
    import tifffile
except ImportError as e:
    raise ImportError(
        "tifffile is required but not installed in this environment. "
        "On Kaggle it should be available by default. Please add it as a notebook dependency."
    ) from e

HAS_IMAGECODECS = False
try:
    import imagecodecs  # optional accelerator for some tifffile paths
    HAS_IMAGECODECS = True
    print("[OK] imagecodecs available")
except Exception:
    print("[INFO] imagecodecs not available (OK)")

from PIL import Image
from scipy import ndimage as ndi

def read_tif_volume(path):
    """Read multipage TIFF into (D,H,W) numpy array."""
    # tifffile is the intended fast path
    try:
        return tifffile.imread(path)
    except Exception:
        # Fallback: PIL multipage (slow). Keep as last resort.
        img = Image.open(path)
        frames = []
        for i in range(getattr(img, "n_frames", 1)):
            img.seek(i)
            frames.append(np.array(img))
        img.close()
        vol = np.stack(frames, axis=0)
        print(f"[WARN] Used PIL fallback for {os.path.basename(path)} (may be slow). shape={vol.shape}")
        return vol

print("[OK] Environment ready")

In [ ]:
# ============================================================
# CELL 2: Imports & Configuration -- Model B
# ============================================================
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from torch.cuda.amp import GradScaler
from torch.utils.checkpoint import checkpoint as grad_checkpoint

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[OK] PyTorch {torch.__version__}  device={DEVICE}")
if DEVICE.type == "cuda":
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"     GPU {i}: {props.name}  VRAM: {props.total_memory / 1e9:.1f} GB")

# NOTE: If Cell 1 already seeds, this re-seeds deterministically for Model B (fine).
SEED = 137  # Different seed from Model A for diversity
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -- Paths (auto-discover) --
_ROOT_CANDIDATES = [
    "/kaggle/input/competitions/vesuvius-challenge-surface-detection",
    "/kaggle/input/vesuvius-challenge-surface-detection",
]
ROOT_DIR = None
for _rc in _ROOT_CANDIDATES:
    if os.path.isdir(_rc) and os.path.isdir(os.path.join(_rc, "train_images")):
        ROOT_DIR = _rc
        break

if ROOT_DIR is None:
    _input = "/kaggle/input"
    if os.path.isdir(_input):
        print(f"[DISCOVER] Contents of {_input}: {os.listdir(_input)}")
        for d in os.listdir(_input):
            fp = os.path.join(_input, d)
            if os.path.isdir(fp) and os.path.isdir(os.path.join(fp, "train_images")):
                ROOT_DIR = fp
                break
            if os.path.isdir(fp):
                for dd in os.listdir(fp):
                    fp2 = os.path.join(fp, dd)
                    if os.path.isdir(fp2) and os.path.isdir(os.path.join(fp2, "train_images")):
                        ROOT_DIR = fp2
                        break

assert ROOT_DIR is not None, "[FATAL] Could not find competition data with train_images/"
print(f"[OK] ROOT_DIR = {ROOT_DIR}")

TRAIN_IMG_DIR  = f"{ROOT_DIR}/train_images"
TRAIN_LBL_DIR  = f"{ROOT_DIR}/train_labels"
CKPT_DIR       = "/kaggle/working/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

# -- Architecture (same as Model A / predict_ensemble) --
NUM_CLASSES       = 2
FEATURES          = [32, 64, 128, 256, 320, 320]
BLOCKS_PER_STAGE  = [1, 3, 4, 6, 6, 6]
STRIDES           = [[1,1,1], [2,2,2], [2,2,2], [2,2,2], [2,2,2], [2,2,2]]
GRAD_CKPT_STAGES  = [3, 4, 5]

# -- Patch size candidates --
PATCH_CANDIDATES  = [(192, 192, 192), (160, 160, 160), (128, 128, 128)]

# -- Training (time budget) --
MAX_TRAIN_HOURS   = 8.5  # keep <= 9h pipeline budget
NUM_ITERATIONS    = 250
TARGET_EPOCHS     = 1000
INITIAL_LR        = 0.01
WEIGHT_DECAY      = 3e-5
GRAD_ACCUM        = 2
EMA_DECAY         = 0.9999
FG_OVERSAMPLE     = 0.67
IGNORE_LABEL      = 255
MAX_TRAIN_VOLS    = 0  # use ALL volumes

# -- Loss weights (Model B: conservative separator) --
W_CE              = 1.0
W_DICE            = 1.0
W_FOCAL           = 0.5
FOCAL_GAMMA       = 2.0
FOCAL_ALPHA_FG    = 0.25  # more conservative

print("[OK] Model B Configuration loaded")
check_budget("Config done")

In [ ]:
# ============================================================
# CELL 3: Discover Training Data (deterministic split)
# ============================================================

import hashlib

train_vol_infos = []
if os.path.isdir(TRAIN_IMG_DIR):
    for f in sorted(os.listdir(TRAIN_IMG_DIR)):
        if f.endswith(".tif"):
            vid = f[:-4]
            img_path = os.path.join(TRAIN_IMG_DIR, f)
            lbl_path = os.path.join(TRAIN_LBL_DIR, f)
            if os.path.exists(lbl_path):
                train_vol_infos.append({"id": vid, "image": img_path, "label": lbl_path})

total_vols = len(train_vol_infos)
assert total_vols > 0, "[FATAL] No training data found (no matching train_images/train_labels .tif pairs)"

# Optional subsample
if MAX_TRAIN_VOLS > 0 and total_vols > MAX_TRAIN_VOLS:
    rng = np.random.RandomState(SEED)
    idx = rng.permutation(total_vols)[:MAX_TRAIN_VOLS]
    train_vol_infos = [train_vol_infos[i] for i in idx]
    print(f"[OK] Subsampled to {MAX_TRAIN_VOLS} from {total_vols}")
else:
    print(f"[OK] Using ALL {total_vols} training volumes")

# Deterministic split by stable hash of volume id (robust to execution order)
def _stable_key(s: str) -> int:
    return int(hashlib.md5(s.encode("utf-8")).hexdigest(), 16)

train_vol_infos = sorted(train_vol_infos, key=lambda x: _stable_key(x["id"]))

# Choose validation size (avoid tiny val sets)
n_val = max(4, len(train_vol_infos) // 10)
n_val = min(n_val, max(1, len(train_vol_infos) - 1))  # ensure at least 1 train

val_vols = train_vol_infos[:n_val]
train_vols = train_vol_infos[n_val:]

print(f"[OK] Train: {len(train_vols)}, Val: {len(val_vols)}")
print("[VAL IDS]", [v["id"] for v in val_vols])
check_budget("Data discovery done")

In [ ]:
# ============================================================
# CELL 4: Data Augmentation (Model B: STRONGER boundary/contrast, topology-safe)
# ============================================================

def augment_3d_patch(vol, lbl):
    """
    Model B augmentation (topology-safe):
    - Stronger contrast/intensity/gamma jitter
    - Noise + anisotropic blur (little/no Z blur)
    - No 3D elastic deformation (too risky for thin sheet topology)
    """
    # Random flips (3 axes, p=0.5 each)
    for axis in range(3):
        if random.random() > 0.5:
            vol = np.flip(vol, axis=axis)
            lbl = np.flip(lbl, axis=axis)

    # Random 90-degree rotation in XY plane
    k = random.randint(0, 3)
    if k > 0:
        vol = np.rot90(vol, k=k, axes=(1, 2))
        lbl = np.rot90(lbl, k=k, axes=(1, 2))

    # Intensity scaling + shift (p=0.8, wider range)
    if random.random() < 0.8:
        scale = random.uniform(0.7, 1.3)
        shift = random.uniform(-0.2, 0.2)
        vol = vol * scale + shift

    # Gamma correction (p=0.6, wider range)
    if random.random() < 0.6:
        v_min, v_max = float(vol.min()), float(vol.max())
        if v_max - v_min > 1e-8:
            vol_01 = (vol - v_min) / (v_max - v_min + 1e-8)
            gamma = random.uniform(0.5, 2.0)
            vol_01 = np.power(np.clip(vol_01, 1e-7, 1.0), gamma)
            vol = vol_01 * (v_max - v_min) + v_min

    # Contrast reduction (p=0.4): forces reliance on geometry
    if random.random() < 0.4:
        mean_val = float(vol.mean())
        factor = random.uniform(0.5, 1.0)
        vol = mean_val + (vol - mean_val) * factor

    # Gaussian noise (p=0.5)
    if random.random() < 0.5:
        sigma = random.uniform(0.01, 0.08)
        vol = vol + np.random.normal(0, sigma, vol.shape).astype(np.float32)

    # Anisotropic Gaussian blur (p=0.35): avoid Z washout
    if random.random() < 0.35:
        s_xy = random.uniform(0.3, 1.5)
        s_z  = random.uniform(0.0, 0.25)  # little/no Z blur
        vol = ndi.gaussian_filter(vol, sigma=(s_z, s_xy, s_xy)).astype(np.float32)

    # Small Z-shift jitter (p=0.25): keeps topology but adds diversity
    if random.random() < 0.25:
        dz = random.randint(-2, 2)
        if dz != 0:
            vol = np.roll(vol, shift=dz, axis=0)
            lbl = np.roll(lbl, shift=dz, axis=0)

    # Robust clipping to prevent extreme outliers
    lo, hi = np.percentile(vol, (0.5, 99.5))
    if hi - lo > 1e-8:
        vol = np.clip(vol, lo, hi)

    return np.ascontiguousarray(vol.astype(np.float32)), np.ascontiguousarray(lbl.astype(np.int64))

print("[OK] Model B augmentation ready (strong boundary/contrast, topology-safe)")

In [ ]:
# ============================================================
# CELL 5: ResidualEncoderUNet (identical to Model A, DS-safe)
# ============================================================

class ResidualBlock3D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv3d(channels, channels, 3, padding=1, bias=False)
        self.norm1 = nn.InstanceNorm3d(channels, eps=1e-5, affine=True)
        self.conv2 = nn.Conv3d(channels, channels, 3, padding=1, bias=False)
        self.norm2 = nn.InstanceNorm3d(channels, eps=1e-5, affine=True)
        self.act = nn.LeakyReLU(0.01, inplace=True)

    def forward(self, x):
        residual = x
        x = self.act(self.norm1(self.conv1(x)))
        x = self.norm2(self.conv2(x))
        return self.act(x + residual)


class EncoderStage(nn.Module):
    def __init__(self, in_ch, out_ch, n_blocks, stride=(1, 1, 1)):
        super().__init__()
        self.initial = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, stride=list(stride), padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, eps=1e-5, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )
        self.blocks = nn.Sequential(*[ResidualBlock3D(out_ch) for _ in range(n_blocks)])

    def forward(self, x):
        return self.blocks(self.initial(x))


class DecoderStage(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.upsample = nn.ConvTranspose3d(in_ch, in_ch, kernel_size=2, stride=2, bias=False)
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, eps=1e-5, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )

    def forward(self, x, skip):
        x = self.upsample(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode="trilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


class ResidualEncoderUNet(nn.Module):
    def __init__(
        self,
        in_ch=1,
        num_classes=2,
        features=(32, 64, 128, 256, 320, 320),
        blocks_per_stage=(1, 3, 4, 6, 6, 6),
        strides=((1,1,1), (2,2,2), (2,2,2), (2,2,2), (2,2,2), (2,2,2)),
        grad_ckpt_stages=None,
    ):
        super().__init__()
        self.grad_ckpt_stages = set(grad_ckpt_stages or [])
        self.features = list(features)
        n_stages = len(self.features)
        assert len(blocks_per_stage) == n_stages
        assert len(strides) == n_stages

        # Encoder
        self.encoders = nn.ModuleList()
        for i in range(n_stages):
            in_c = in_ch if i == 0 else self.features[i - 1]
            self.encoders.append(EncoderStage(in_c, self.features[i], blocks_per_stage[i], stride=strides[i]))

        # Decoder outputs channels = features[i] for i in [n_stages-2 ... 0]
        self.decoders = nn.ModuleList()
        for i in range(n_stages - 2, -1, -1):
            self.decoders.append(DecoderStage(self.features[i + 1], self.features[i], self.features[i]))

        # Deep supervision heads on each decoder output (finest -> coarsest)
        # decoder_outputs[j] has channels = features[j], j in [0..n_stages-2]
        self.seg_heads = nn.ModuleList([nn.Conv3d(self.features[i], num_classes, 1) for i in range(n_stages - 1)])

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv3d, nn.ConvTranspose3d)):
                nn.init.kaiming_normal_(m.weight, a=0.01, mode="fan_out", nonlinearity="leaky_relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x, deep_supervision: bool = False):
        skips = []
        for i, enc in enumerate(self.encoders):
            if self.training and i in self.grad_ckpt_stages:
                x = grad_checkpoint(enc, x, use_reentrant=False)
            else:
                x = enc(x)
            skips.append(x)

        x = skips[-1]
        decoder_outputs = []
        for i, dec in enumerate(self.decoders):
            x = dec(x, skips[len(skips) - 2 - i])
            decoder_outputs.append(x)

        decoder_outputs = decoder_outputs[::-1]  # finest -> coarsest

        # Main output at finest scale
        out = self.seg_heads[0](decoder_outputs[0])

        if deep_supervision and self.training:
            ds_outputs = [out]
            target_size = out.shape[2:]
            # Auxiliary outputs (upsampled to target_size)
            for i in range(1, len(decoder_outputs)):
                assert decoder_outputs[i].shape[1] == self.features[i], "Channel mismatch in DS head wiring"
                ds_logit = self.seg_heads[i](decoder_outputs[i])
                ds_logit = F.interpolate(ds_logit, size=target_size, mode="trilinear", align_corners=False)
                ds_outputs.append(ds_logit)
            return tuple(ds_outputs)

        return out


# Parameter count sanity
_tmp = ResidualEncoderUNet(
    features=FEATURES,
    blocks_per_stage=BLOCKS_PER_STAGE,
    strides=[tuple(s) for s in STRIDES],
    grad_ckpt_stages=GRAD_CKPT_STAGES,
)
n_params = sum(p.numel() for p in _tmp.parameters()) / 1e6
del _tmp
print(f"[OK] ResidualEncoderUNet: {n_params:.2f}M parameters")
gc.collect()
check_budget("Architecture defined")

In [ ]:
# ============================================================
# CELL 6: Patch Size Auto-Detection (robust)
# ============================================================

def find_max_patch_size(candidates, device):
    # Ensure largest -> smallest by voxel count
    candidates = sorted(candidates, key=lambda p: p[0]*p[1]*p[2], reverse=True)

    for patch in candidates:
        model = x = out = loss = None
        try:
            if device.type == "cuda":
                torch.cuda.empty_cache()
                gc.collect()
                torch.cuda.reset_peak_memory_stats(device)

            model = ResidualEncoderUNet(
                in_ch=1, num_classes=NUM_CLASSES,
                features=FEATURES, blocks_per_stage=BLOCKS_PER_STAGE,
                strides=[tuple(s) for s in STRIDES],
                grad_ckpt_stages=GRAD_CKPT_STAGES,
            ).to(device)
            model.train()

            x = torch.randn(1, 1, *patch, device=device)

            if device.type == "cuda":
                with autocast("cuda"):
                    out = model(x, deep_supervision=True)
                    loss = out[0].mean() if isinstance(out, tuple) else out.mean()
            else:
                out = model(x, deep_supervision=True)
                loss = out[0].mean() if isinstance(out, tuple) else out.mean()

            loss.backward()

            if device.type == "cuda":
                vram = torch.cuda.max_memory_allocated(device) / 1e9
                print(f"[OK] Patch {patch} fits -- peak VRAM: {vram:.2f} GB")
            else:
                print(f"[OK] Patch {patch} fits on CPU (VRAM stats n/a)")

            # cleanup
            del model, x, out, loss
            gc.collect()
            if device.type == "cuda":
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats(device)

            return patch

        except RuntimeError as e:
            msg = str(e).lower()
            is_oom = ("out of memory" in msg) or ("cuda out of memory" in msg)
            if is_oom and device.type == "cuda":
                print(f"[OOM] Patch {patch} too large, trying smaller...")
                try:
                    del model, x, out, loss
                except Exception:
                    pass
                gc.collect()
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats(device)
                continue
            raise

    raise RuntimeError("[FATAL] Even smallest patch causes OOM")

TRAIN_PATCH = find_max_patch_size(PATCH_CANDIDATES, DEVICE)
print(f"\n[OK] Selected training patch: {TRAIN_PATCH}")
check_budget("Patch size selected")

In [ ]:
# ============================================================
# CELL 7: Loss Functions (Model B: CE + Dice + Focal)
# Goal: Conservative separator (avoid over-connecting / mergers)
# Note: In focal loss, alpha typically weights the FOREGROUND class.
#       For conservative predictions, use alpha_fg < 0.5.
# ============================================================

class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, target):
        valid = (target != IGNORE_LABEL)
        if not valid.any():
            return logits.sum() * 0.0

        target_clean = target.clone()
        target_clean[~valid] = 0

        probs = torch.softmax(logits, dim=1)
        # one-hot: (B,D,H,W,C) -> (B,C,D,H,W)
        target_oh = F.one_hot(target_clean, logits.shape[1]).permute(0, 4, 1, 2, 3).float()
        valid_5d = valid.unsqueeze(1).float()

        probs_m = probs * valid_5d
        target_m = target_oh * valid_5d

        dims = (0, 2, 3, 4)
        inter = (probs_m * target_m).sum(dims)
        union = probs_m.sum(dims) + target_m.sum(dims)
        dice = 1.0 - (2.0 * inter + self.smooth) / (union + self.smooth)
        return dice.mean()


class FocalLoss(nn.Module):
    """
    Multi-class focal loss (2-class here).
    alpha_fg weights the FOREGROUND class (target==1). For conservative predictions,
    set alpha_fg < 0.5 (e.g., 0.25).
    """
    def __init__(self, gamma=2.0, alpha_fg=0.25, ignore_index=255):
        super().__init__()
        self.gamma = gamma
        self.alpha_fg = alpha_fg
        self.ignore_index = ignore_index

    def forward(self, logits, target):
        valid = (target != self.ignore_index)
        if not valid.any():
            return logits.sum() * 0.0

        # Flatten valid voxels: logits_v (N,C), target_v (N,)
        logits_v = logits.permute(0, 2, 3, 4, 1)[valid]
        target_v = target[valid]

        log_p = F.log_softmax(logits_v, dim=1)
        p = torch.exp(log_p)

        ce = F.nll_loss(log_p, target_v, reduction="none")
        p_t = p.gather(1, target_v.unsqueeze(1)).squeeze(1)

        focal_weight = (1.0 - p_t).clamp_min(0.0) ** self.gamma

        # alpha for FG, (1-alpha) for BG (since 2-class)
        alpha_t = torch.where(target_v == 1, torch.tensor(self.alpha_fg, device=logits.device),
                              torch.tensor(1.0 - self.alpha_fg, device=logits.device))

        loss = alpha_t * focal_weight * ce
        return loss.mean()


class ModelBLoss(nn.Module):
    """Model B: CE + Dice + Focal (conservative separator)."""
    def __init__(self, w_ce=1.0, w_dice=1.0, w_focal=0.5,
                 focal_gamma=2.0, focal_alpha_fg=0.25):
        super().__init__()
        self.w_ce = w_ce
        self.w_dice = w_dice
        self.w_focal = w_focal
        self.ce = nn.CrossEntropyLoss(ignore_index=IGNORE_LABEL)
        self.dice = DiceLoss()
        self.focal = FocalLoss(gamma=focal_gamma, alpha_fg=focal_alpha_fg, ignore_index=IGNORE_LABEL)

    def forward(self, logits, target):
        ce_loss = self.ce(logits, target)
        dice_loss = self.dice(logits, target)
        focal_loss = self.focal(logits, target)

        total = self.w_ce * ce_loss + self.w_dice * dice_loss + self.w_focal * focal_loss

        # Safe guard: keep graph connected
        if torch.isnan(total) or torch.isinf(total):
            return logits.sum() * 0.0
        return total


print("[OK] Model B loss ready (CE + Dice + Focal, conservative)")
print("      NOTE: focal_alpha_fg < 0.5 => conservative; recommend 0.25")

In [ ]:
# ============================================================
# CELL 8: Dataset (B) — Boundary+FG+BG tri-sampler (more BG = conservative)
# ============================================================
import numpy as np
import random
import scipy.ndimage as ndi
from torch.utils.data import Dataset

IGNORE_LABEL = 255

P_BOUNDARY = 0.40
P_FG       = 0.35
P_BG       = 0.25   # higher BG makes B more conservative

MAX_TRIES  = 48

def _safe_coords(mask: np.ndarray):
    coords = np.argwhere(mask)
    if coords.size == 0:
        return None
    return coords

def _pick_center(coords, shape):
    c = coords[random.randrange(len(coords))]
    D,H,W = shape
    return (int(np.clip(c[0], 0, D-1)), int(np.clip(c[1], 0, H-1)), int(np.clip(c[2], 0, W-1)))

class VesuviusPatchDataset(Dataset):
    def __init__(self, vol_infos, patch_size=(192,192,192),
                 num_iterations=250, fg_rate=0.67, augment=True,
                 p_boundary=P_BOUNDARY, p_fg=P_FG, p_bg=P_BG,
                 boundary_dilate=2):
        self.vol_infos = vol_infos
        self.ps = tuple(patch_size)
        self.length = int(num_iterations)
        self.augment = bool(augment)

        self.p_boundary = float(p_boundary)
        self.p_fg = float(p_fg)
        self.p_bg = float(p_bg)
        self.boundary_dilate = int(boundary_dilate)

        self._cached_vid = None
        self._cached_vol = None
        self._cached_lbl = None
        self._fg_coords = None
        self._bd_coords = None
        self._bg_coords = None

        print(f"[DATASET-B] vols={len(vol_infos)} iters/epoch={self.length} patch={self.ps} "
              f"mix(boundary/fg/bg)={self.p_boundary:.2f}/{self.p_fg:.2f}/{self.p_bg:.2f}")

    def __len__(self):
        return self.length

    def _load(self, vid, img_path, lbl_path):
        if self._cached_vid == vid and self._cached_vol is not None:
            return self._cached_vol, self._cached_lbl

        vol = read_tif_volume(img_path).astype(np.float32)
        lbl = read_tif_volume(lbl_path).astype(np.uint8)

        m = vol > 0
        if m.any():
            mu = float(vol[m].mean()); sd = float(vol[m].std() + 1e-6)
            vol[m] = (vol[m] - mu) / sd
        vol[~m] = 0.0

        lbl = lbl.copy()
        lbl[lbl == 2] = IGNORE_LABEL
        valid = (lbl != IGNORE_LABEL)
        fg = valid & (lbl == 1)

        if fg.any():
            dil = ndi.binary_dilation(fg, iterations=self.boundary_dilate)
            ero = ndi.binary_erosion(fg, iterations=1)
            bd = (dil ^ ero) & valid
        else:
            bd = np.zeros_like(fg, dtype=bool)

        bg = valid & (~fg)

        self._fg_coords = _safe_coords(fg)
        self._bd_coords = _safe_coords(bd)
        self._bg_coords = _safe_coords(bg)

        self._cached_vid = vid
        self._cached_vol = vol
        self._cached_lbl = lbl
        return vol, lbl

    def _crop_around(self, vol, lbl, center):
        D,H,W = vol.shape
        pd,ph,pw = self.ps
        cd,ch,cw = center
        d0 = int(np.clip(cd - pd//2, 0, max(0, D - pd)))
        h0 = int(np.clip(ch - ph//2, 0, max(0, H - ph)))
        w0 = int(np.clip(cw - pw//2, 0, max(0, W - pw)))
        return vol[d0:d0+pd, h0:h0+ph, w0:w0+pw], lbl[d0:d0+pd, h0:h0+ph, w0:w0+pw]

    def __getitem__(self, i):
        vid, img_path, lbl_path = self.vol_infos[random.randrange(len(self.vol_infos))]
        vol, lbl = self._load(vid, img_path, lbl_path)

        r = random.random()
        coords = self._bg_coords
        if r < self.p_boundary and self._bd_coords is not None:
            coords = self._bd_coords
        elif r < (self.p_boundary + self.p_fg) and self._fg_coords is not None:
            coords = self._fg_coords

        if coords is None:
            D,H,W = vol.shape
            pd,ph,pw = self.ps
            d0 = random.randint(0, max(0, D - pd))
            h0 = random.randint(0, max(0, H - ph))
            w0 = random.randint(0, max(0, W - pw))
            pv = vol[d0:d0+pd, h0:h0+ph, w0:w0+pw]
            pl = lbl[d0:d0+pd, h0:h0+ph, w0:w0+pw]
        else:
            pv, pl = self._crop_around(vol, lbl, _pick_center(coords, vol.shape))

        if self.augment:
            pv, pl = augment_3d_patch(pv, pl)

        x = torch.from_numpy(np.ascontiguousarray(pv[None])).float()
        y = torch.from_numpy(np.ascontiguousarray(pl)).long()
        return x, y

print("[OK] Dataset B ready (boundary/fg/bg tri-sampler)")

In [ ]:
# ============================================================
# CELL 8.5: SANITY PROBE (Model B) — run BEFORE training
# Insert between CELL 8 (Dataset) and CELL 9 (Training Loop)
# ============================================================
import numpy as np
import scipy.ndimage as ndi

# Build a small probe dataset OUTSIDE the training function
train_dataset = VesuviusPatchDataset(
    train_vols,
    patch_size=TRAIN_PATCH,
    num_iterations=min(32, NUM_ITERATIONS),
    fg_rate=FG_OVERSAMPLE,
    augment=True,
)
print("[OK] Created probe train_dataset for sanity-check:", type(train_dataset))

def _describe_mask(m, name="mask"):
    m = (m > 0).astype(np.uint8)
    fg = int(m.sum())
    tot = int(m.size)
    fg_pct = 100.0 * fg / max(1, tot)
    struct26 = ndi.generate_binary_structure(3, 3)
    cc, ncc = ndi.label(m, structure=struct26)
    if ncc > 0:
        sizes = np.bincount(cc.ravel())[1:]
        largest = int(sizes.max())
        largest_pct = 100.0 * largest / max(1, fg)
    else:
        largest_pct = 0.0
    print(f"[{name}] FG%={fg_pct:.4f}% | nCC(26)={ncc} | largest_share={largest_pct:.2f}%")

def _extract_y(sample):
    if isinstance(sample, (tuple, list)):
        y = sample[1]
    elif isinstance(sample, dict):
        y = sample.get("y", sample.get("label", sample.get("lbl", None)))
        if y is None:
            raise ValueError(f"Sample dict keys: {list(sample.keys())}")
    else:
        raise ValueError(f"Unsupported sample type: {type(sample)}")

    if hasattr(y, "detach"):
        y = y.detach().cpu().numpy()
    y = np.asarray(y)
    if y.ndim == 4 and y.shape[0] == 1:
        y = y[0]
    y = np.squeeze(y)
    if y.ndim != 3:
        raise ValueError(f"Label patch must be 3D, got {y.shape}")

    has_2 = bool((y == 2).any())
    ignore = int((y == 255).sum())
    uniq = set(np.unique(y).tolist())
    return y.astype(np.int32), ignore, has_2, uniq

n = min(8, len(train_dataset))
fg_pcts, ign_pcts, nccs = [], [], []

print(f"[PROBE] Sampling {n} patches...")
for i in range(n):
    y, ign, has_2, uniq = _extract_y(train_dataset[np.random.randint(0, len(train_dataset))])

    if has_2:
        raise RuntimeError("[FATAL] Found label==2 in patches. You did NOT map 2->255 correctly.")
    if not uniq.issubset({0,1,255}):
        raise RuntimeError(f"[FATAL] Unexpected label values {uniq}. Expected subset of {{0,1,255}}.")

    fg = (y == 1).astype(np.uint8)
    _describe_mask(fg, name=f"patch{i:02d}")

    fg_pcts.append(100.0 * float(fg.mean()))
    ign_pcts.append(100.0 * float(ign) / float(y.size))

    cc, ncc = ndi.label(fg, structure=ndi.generate_binary_structure(3,3))
    nccs.append(int(ncc))

print("\n[SUMMARY]")
print(f"  FG%:     mean={np.mean(fg_pcts):.4f}  min={np.min(fg_pcts):.4f}  max={np.max(fg_pcts):.4f}")
print(f"  IGN%:    mean={np.mean(ign_pcts):.4f}  min={np.min(ign_pcts):.4f}  max={np.max(ign_pcts):.4f}")
print(f"  nCC(26): mean={np.mean(nccs):.2f}    min={np.min(nccs)}    max={np.max(nccs)}")

if np.mean(fg_pcts) < 0.02:
    print("[WARN] FG% extremely low → sampler might be too empty-heavy.")
if np.mean(nccs) > 5:
    print("[WARN] Many components in patches → fragmentation risk (VOI_split/Topo k=0).")

print("\n✅ Sanity probe passed. Proceed to training.")

In [ ]:
# ============================================================
# SHARED VALIDATION UTILITY (Insert between Cell 9 and Cell 10)
# Full-volume leaderboard-aligned validation:
# - SurfaceDice@tau (tau=2.0)
# - VOI_score (alpha=0.3) via 26-connectivity CC labelings
# - Topology proxies (splits + cavities) for TopoScore correlation
# ============================================================

import numpy as np
import torch
import scipy.ndimage as ndi

VAL_TAU = 2.0
VOI_ALPHA = 0.3
CC_CONN = 3  # 26-connectivity via generate_binary_structure(3,3)

def _softmax_np(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x, dtype=np.float32)
    return e / (np.sum(e, axis=axis, keepdims=True) + 1e-8)

def _gaussian_weight_map(shape_zyx, sigma_scale=0.125):
    D, H, W = shape_zyx
    zz = np.linspace(-1, 1, D, dtype=np.float32)
    yy = np.linspace(-1, 1, H, dtype=np.float32)
    xx = np.linspace(-1, 1, W, dtype=np.float32)
    Z, Y, X = np.meshgrid(zz, yy, xx, indexing="ij")
    r2 = Z*Z + Y*Y + X*X
    sigma2 = (sigma_scale ** 2)
    w = np.exp(-0.5 * r2 / max(sigma2, 1e-6)).astype(np.float32)
    return w / (w.max() + 1e-8)

@torch.no_grad()
def svu_sliding_window_logits(model, vol_zyx, roi_size=(64,192,192), overlap=0.35, num_classes=2):
    """
    vol_zyx: np.ndarray float32 (D,H,W)
    returns logits_zyxc: np.ndarray float32 (D,H,W,C)
    """
    model.eval()
    D, H, W = vol_zyx.shape
    rz, ry, rx = roi_size

    sz = max(1, int(rz * (1.0 - overlap)))
    sy = max(1, int(ry * (1.0 - overlap)))
    sx = max(1, int(rx * (1.0 - overlap)))

    z_starts = list(range(0, max(D - rz, 0) + 1, sz)) or [0]
    y_starts = list(range(0, max(H - ry, 0) + 1, sy)) or [0]
    x_starts = list(range(0, max(W - rx, 0) + 1, sx)) or [0]
    if D > rz and z_starts[-1] != D - rz: z_starts.append(D - rz)
    if H > ry and y_starts[-1] != H - ry: y_starts.append(H - ry)
    if W > rx and x_starts[-1] != W - rx: x_starts.append(W - rx)

    out = np.zeros((D, H, W, num_classes), dtype=np.float32)
    wsum = np.zeros((D, H, W, 1), dtype=np.float32)

    w_patch = _gaussian_weight_map((min(rz, D), min(ry, H), min(rx, W)))[..., None]

    for z0 in z_starts:
        for y0 in y_starts:
            for x0 in x_starts:
                z1 = min(z0 + rz, D); y1 = min(y0 + ry, H); x1 = min(x0 + rx, W)
                patch = vol_zyx[z0:z1, y0:y1, x0:x1].astype(np.float32, copy=False)

                padz = rz - patch.shape[0]; pady = ry - patch.shape[1]; padx = rx - patch.shape[2]
                if padz or pady or padx:
                    patch = np.pad(patch, ((0,padz),(0,pady),(0,padx)), mode="reflect")

                inp = torch.from_numpy(patch[None, None]).to(DEVICE, non_blocking=True)  # (1,1,rz,ry,rx)
                logits = model(inp)  # (1,C,rz,ry,rx)
                logits = logits.float().cpu().numpy()[0].transpose(1,2,3,0)  # (rz,ry,rx,C)

                logits = logits[:(z1-z0), :(y1-y0), :(x1-x0), :]
                wp = w_patch[:(z1-z0), :(y1-y0), :(x1-x0), :]

                out[z0:z1, y0:y1, x0:x1, :] += logits * wp
                wsum[z0:z1, y0:y1, x0:x1, :] += wp

    out = out / np.clip(wsum, 1e-6, None)
    return out

def svu_hysteresis_bin(prob, tl, th):
    strong = prob >= th
    weak = prob >= tl
    struct26 = ndi.generate_binary_structure(3, CC_CONN)
    lbl, n = ndi.label(weak, structure=struct26)
    if n == 0:
        return np.zeros_like(prob, dtype=np.uint8)
    strong_ids = np.unique(lbl[strong])
    strong_ids = strong_ids[strong_ids != 0]
    keep = np.isin(lbl, strong_ids)
    return keep.astype(np.uint8)

def svu_surface_dice_at_tau(pred_bin, gt_bin, spacing=(1.0,1.0,1.0), tau=2.0):
    pred = (pred_bin > 0)
    gt = (gt_bin > 0)

    if pred.sum() == 0 and gt.sum() == 0: return 1.0
    if (pred.sum() == 0) ^ (gt.sum() == 0): return 0.0

    st = ndi.generate_binary_structure(3, 1)
    pred_s = pred & ~ndi.binary_erosion(pred, structure=st, iterations=1, border_value=0)
    gt_s   = gt   & ~ndi.binary_erosion(gt,   structure=st, iterations=1, border_value=0)

    dt_gt = ndi.distance_transform_edt(~gt_s, sampling=spacing)
    dt_pr = ndi.distance_transform_edt(~pred_s, sampling=spacing)

    p2g = (dt_gt[pred_s] <= tau).mean() if pred_s.any() else 1.0
    g2p = (dt_pr[gt_s]   <= tau).mean() if gt_s.any() else 1.0
    return float(0.5 * (p2g + g2p))

def svu_voi_score(pred_bin, gt_bin, alpha=0.3):
    pred = (pred_bin > 0)
    gt   = (gt_bin > 0)

    if pred.sum() == 0 and gt.sum() == 0: return 1.0
    if (pred.sum() == 0) ^ (gt.sum() == 0): return 0.0

    struct26 = ndi.generate_binary_structure(3, CC_CONN)
    pred_cc, _ = ndi.label(pred, structure=struct26)
    gt_cc, _   = ndi.label(gt,   structure=struct26)

    u = pred | gt
    p = pred_cc[u].astype(np.int64, copy=False)
    g = gt_cc[u].astype(np.int64, copy=False)

    _, p = np.unique(p, return_inverse=True)
    _, g = np.unique(g, return_inverse=True)

    n = p.size
    if n == 0: return 1.0

    Pg = p.max() + 1
    Gg = g.max() + 1
    idx = p * Gg + g
    c = np.bincount(idx, minlength=Pg*Gg).astype(np.float64).reshape(Pg, Gg)

    P = c / n
    p_m = P.sum(axis=1, keepdims=True)
    g_m = P.sum(axis=0, keepdims=True)

    eps = 1e-12
    voi_split = -np.sum(P * (np.log(P + eps) - np.log(p_m + eps)))
    voi_merge = -np.sum(P * (np.log(P + eps) - np.log(g_m + eps)))
    voi_total = float(voi_split + voi_merge)

    return float(1.0 / (1.0 + alpha * voi_total))

def svu_topo_proxies(pred_bin, gt_bin):
    struct26 = ndi.generate_binary_structure(3, CC_CONN)
    pred = (pred_bin > 0)
    gt   = (gt_bin > 0)

    _, pred_n = ndi.label(pred, structure=struct26)
    _, gt_n   = ndi.label(gt,   structure=struct26)

    fg = pred | gt
    cavities = 0
    if fg.any():
        coords = np.argwhere(fg)
        z0,y0,x0 = coords.min(axis=0); z1,y1,x1 = coords.max(axis=0) + 1
        pad = 5
        z0 = max(0, z0-pad); y0=max(0,y0-pad); x0=max(0,x0-pad)
        z1 = min(fg.shape[0], z1+pad); y1=min(fg.shape[1], y1+pad); x1=min(fg.shape[2], x1+pad)

        roi_fg = fg[z0:z1, y0:y1, x0:x1]
        roi_bg = ~roi_fg
        bg_cc, bg_n = ndi.label(roi_bg, structure=struct26)

        touch = np.zeros(bg_n + 1, dtype=bool)
        touch[np.unique(bg_cc[0,:,:])] = True
        touch[np.unique(bg_cc[-1,:,:])] = True
        touch[np.unique(bg_cc[:,0,:])] = True
        touch[np.unique(bg_cc[:,-1,:])] = True
        touch[np.unique(bg_cc[:,:,0])] = True
        touch[np.unique(bg_cc[:,:,-1])] = True

        cavities = sum((not touch[i]) for i in range(1, bg_n+1))

    return {"pred_components": int(pred_n), "gt_components": int(gt_n), "cavities_proxy": int(cavities)}

@torch.no_grad()
def svu_validate_full_volume_case(model, img_path, lbl_path, roi_size, overlap, tl, th, spacing=(1.0,1.0,1.0)):
    vol = read_tif_volume(img_path).astype(np.float32)
    gt  = read_tif_volume(lbl_path).astype(np.uint8)

    gt[gt == 2] = IGNORE_LABEL
    valid = (gt != IGNORE_LABEL)
    gt_bin = ((gt == 1) & valid).astype(np.uint8)

    logits = svu_sliding_window_logits(model, vol, roi_size=roi_size, overlap=overlap, num_classes=NUM_CLASSES)
    prob = _softmax_np(logits, axis=-1)[..., 1].astype(np.float32)

    pred_bin = svu_hysteresis_bin(prob, tl, th)
    pred_bin = (pred_bin & valid).astype(np.uint8)

    sd  = svu_surface_dice_at_tau(pred_bin, gt_bin, spacing=spacing, tau=VAL_TAU)
    voi = svu_voi_score(pred_bin, gt_bin, alpha=VOI_ALPHA)
    tp  = svu_topo_proxies(pred_bin, gt_bin)

    return {"surface_dice_tau": float(sd), "voi_score": float(voi), **tp}

def svu_combined_val_score(m):
    sd  = m["surface_dice_tau"]
    voi = m["voi_score"]

    split_pen = min(1.0, abs(m["pred_components"] - m["gt_components"]) / max(1, m["gt_components"]))
    split_proxy = 1.0 - split_pen

    cav = m["cavities_proxy"]
    cav_proxy = 1.0 / (1.0 + 0.25 * cav)

    topo_proxy = 0.5 * split_proxy + 0.5 * cav_proxy
    return float(0.35*sd + 0.35*voi + 0.30*topo_proxy)

print("[OK] Shared full-volume validation utilities ready")

In [ ]:
# ============================================================
# CELL 9: Training Loop (Model B) — FIXED VALIDATION + BEST BY VAL
# ============================================================
from contextlib import nullcontext
import numpy as np
import torch
from torch.utils.data import DataLoader

class EMAModel:
    def __init__(self, model, decay=0.9999):
        self.decay = float(decay)
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}
        self._backup = None

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, v in msd.items():
            self.shadow[k].mul_(self.decay).add_(v, alpha=1.0 - self.decay)

    @torch.no_grad()
    def store(self, model):
        self._backup = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def apply(self, model):
        model.load_state_dict(self.shadow, strict=True)

    @torch.no_grad()
    def restore(self, model):
        if self._backup is None:
            return
        model.load_state_dict(self._backup, strict=True)
        self._backup = None

def poly_lr(epoch, max_epoch, initial_lr, exponent=0.9):
    if max_epoch <= 1:
        return initial_lr
    return initial_lr * (1 - epoch / max_epoch) ** exponent

def _softmax_np(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x, dtype=np.float32)
    return e / (np.sum(e, axis=axis, keepdims=True) + 1e-8)

def build_fixed_val_cache(val_vols, n_patches=512, seed=123):
    ds = VesuviusPatchDataset(
        val_vols,
        patch_size=TRAIN_PATCH,
        num_iterations=n_patches,
        fg_rate=0.0,      # IMPORTANT: no oversampling for honest val distribution
        augment=False,
    )
    cache = []
    for i in range(n_patches):
        x, y = ds[i]
        if x.ndim == 3: x = x[None]
        if y.ndim == 3: y = y[None]
        cache.append((x.contiguous(), y.contiguous()))
    return cache

@torch.no_grad()
def validate_on_cache(model, cache):
    model.eval()
    dices = []
    for x, y in cache:
        inp = x[None].to(DEVICE, non_blocking=True)
        out = model(inp).float().cpu().numpy()[0].transpose(1,2,3,0)
        p = _softmax_np(out, axis=-1)[..., 1]
        pred = (p >= 0.5).astype(np.uint8)
        gt = (y.numpy()[0] > 0.5).astype(np.uint8)
        inter = (pred & gt).sum()
        denom = pred.sum() + gt.sum()
        dice = (2.0 * inter / denom) if denom > 0 else 1.0
        dices.append(float(dice))
    model.train()
    return float(np.mean(dices))

def train_model_b():
    print(f"\n{'='*60}")
    print(f"[TRAIN] Model B -- Conservative Separator (CE + Dice + Focal)")
    print(f"  Patch: {TRAIN_PATCH}, LR: {INITIAL_LR}")
    print(f"  Focal: gamma={FOCAL_GAMMA}, alpha_fg={FOCAL_ALPHA_FG}")
    print(f"{'='*60}\n")

    model = ResidualEncoderUNet(
        in_ch=1, num_classes=NUM_CLASSES,
        features=FEATURES, blocks_per_stage=BLOCKS_PER_STAGE,
        strides=[tuple(s) for s in STRIDES],
        grad_ckpt_stages=GRAD_CKPT_STAGES,
    ).to(DEVICE)

    criterion = ModelBLoss(
        w_ce=W_CE, w_dice=W_DICE, w_focal=W_FOCAL,
        focal_gamma=FOCAL_GAMMA, focal_alpha_fg=FOCAL_ALPHA_FG
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=INITIAL_LR, weight_decay=WEIGHT_DECAY)
    scaler = GradScaler("cuda") if (DEVICE.type == "cuda") else None
    ema = EMAModel(model, decay=EMA_DECAY)

    ds = VesuviusPatchDataset(
        train_vols, patch_size=TRAIN_PATCH,
        num_iterations=NUM_ITERATIONS,
        fg_rate=FG_OVERSAMPLE, augment=True,
    )
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)

    VAL_PATCHES = int(min(1024, max(256, NUM_ITERATIONS // 4)))
    val_cache = build_fixed_val_cache(val_vols, n_patches=VAL_PATCHES, seed=SEED + 1999)
    print(f"[VAL] Fixed cache built: {len(val_cache)} patches")

    best_val = -1.0
    losses = []
    train_start = time.time()

    for epoch in range(TARGET_EPOCHS):
        if (time.time() - train_start) / 3600.0 > MAX_TRAIN_HOURS:
            print("[BUDGET] Max train hours reached. Stopping.")
            break

        # Poly LR
        lr = poly_lr(epoch, TARGET_EPOCHS, INITIAL_LR, exponent=0.9)
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        model.train()
        running = 0.0
        n = 0

        optimizer.zero_grad(set_to_none=True)

        for it in range(NUM_ITERATIONS):
            x, y = ds[it]
            if x.ndim == 3: x = x[None]
            if y.ndim == 3: y = y[None]
            inp = x[None].to(DEVICE, non_blocking=True)
            tgt = y[None].to(DEVICE, non_blocking=True)

            ctx = torch.cuda.amp.autocast(enabled=(scaler is not None))
            with ctx:
                out = model(inp)
                loss = criterion(out, tgt)

            if scaler is not None:
                scaler.scale(loss / GRAD_ACCUM).backward()
            else:
                (loss / GRAD_ACCUM).backward()

            if ((it + 1) % GRAD_ACCUM) == 0:
                if scaler is not None:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                ema.update(model)

            running += float(loss.detach().cpu().item())
            n += 1

            del inp, tgt, out, loss
            if DEVICE.type == "cuda" and ((it + 1) % 64 == 0):
                torch.cuda.empty_cache()

        avg_loss = running / max(1, n)
        losses.append(avg_loss)

        # Validate on EMA weights
        ema.store(model)
        ema.apply(model)
        val_dice = validate_on_cache(model, val_cache)
        print(f"[E{epoch+1:02d}] train_loss={avg_loss:.4f} | val_dice@0.5={val_dice:.4f} | lr={lr:.2e}")

        # --- Full-volume leaderboard-aligned check (every VAL_EVERY epochs) ---
        VAL_EVERY = 2
        VAL_N_FULL = 1
        FULL_OVERLAP = 0.35
        FULL_ROI = (TRAIN_PATCH[0], TRAIN_PATCH[1], TRAIN_PATCH[2])  # same as training patch
        TL_TH = (0.55, 0.85)

        if ((epoch + 1) % VAL_EVERY) == 0:
            full_cases = val_vol_infos[:VAL_N_FULL]
            full_metrics = []
            for img_path, _, lbl_path in full_cases:
                m = svu_validate_full_volume_case(
                    model, img_path, lbl_path,
                    roi_size=FULL_ROI, overlap=FULL_OVERLAP,
                    tl=TL_TH[0], th=TL_TH[1],
                    spacing=(1.0,1.0,1.0),
                )
                full_metrics.append(m)
            mean_combo = float(np.mean([svu_combined_val_score(mm) for mm in full_metrics]))
            print(f"[FULLVAL] mean_combo={mean_combo:.4f} | details={full_metrics[0] if full_metrics else None}")

            if ("best_combo" not in locals()) or (mean_combo > best_combo):
                best_combo = mean_combo
                torch.save(model.state_dict(), f"{CKPT_DIR}/model_b_best_combo.pt")
                print(f"[SAVE] New best COMBO={best_combo:.4f} -> {CKPT_DIR}/model_b_best_combo.pt")
        
        if val_dice > best_val:
            best_val = val_dice
            torch.save(model.state_dict(), f"{CKPT_DIR}/model_b_best.pt")
            print(f"[SAVE] New best val={best_val:.4f} -> {CKPT_DIR}/model_b_best.pt")

        ema.restore(model)
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    return model, losses

In [ ]:
# ============================================================
# CELL 10: Execute Training
# ============================================================
check_budget("Training start")

try:
    model_b, losses_b = train_model_b()
except Exception as e:
    check_budget("Training crashed")
    raise

gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

check_budget("Training complete")

In [ ]:
# ============================================================
# CELL 11: Threshold Calibration (Model B) — META ONLY (no inference)
# Replaces existing Cell 12 in 2_train_model_b.ipynb
# ============================================================

import os, json
import numpy as np

PP_VERSION = "vesu_pp_v2"
MODEL_TAG = "b"

def fg_fraction_from_lbl(lbl):
    valid = (lbl != IGNORE_LABEL)
    if not valid.any(): return 0.0
    return float(((lbl == 1) & valid).sum() / (valid.sum() + 1e-12))

val_fracs = []
for _, _, lbl_path in val_vol_infos:
    gt = read_tif_volume(lbl_path).astype(np.uint8)
    gt[gt == 2] = IGNORE_LABEL
    val_fracs.append(fg_fraction_from_lbl(gt))

target_fg_med = float(np.median(val_fracs))
target_fg_lo  = float(np.quantile(val_fracs, 0.25))
target_fg_hi  = float(np.quantile(val_fracs, 0.75))

TH_PACK = [
    {"tl": 0.55, "th": 0.85},
    {"tl": 0.50, "th": 0.80},
    {"tl": 0.60, "th": 0.88},
]

meta = {
    "pp_version": PP_VERSION,
    "model_tag": "B",
    "target_fg_med": target_fg_med,
    "target_fg_lo": target_fg_lo,
    "target_fg_hi": target_fg_hi,
    "th_pack": TH_PACK,
}

os.makedirs(CKPT_DIR, exist_ok=True)
meta_path = os.path.join(CKPT_DIR, "model_b_meta.json")
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[OK] Wrote {meta_path}")
print(f"[CAL-B] FG median={target_fg_med:.6f} IQR=({target_fg_lo:.6f},{target_fg_hi:.6f}) | pack={len(TH_PACK)}")

In [ ]:
# ============================================================
# Canonicalize Model B checkpoint (ensemble-proof)
#   - Ensures model_b.pt exists AND is saved as {"state_dict": ...}
# ============================================================
import os, glob, time, torch, json

os.makedirs(CKPT_DIR, exist_ok=True)

def _load_state_dict_any(path):
    obj = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(obj, dict) and "state_dict" in obj and isinstance(obj["state_dict"], dict):
        return obj["state_dict"], obj
    if isinstance(obj, dict) and all(hasattr(v, "shape") for v in obj.values()):
        return obj, {"raw_state_dict": True}
    raise ValueError(f"Unrecognized checkpoint format: {path}")

def _wrap_and_save(path_out, state_dict, epoch=None, extra=None):
    ckpt = {
        "state_dict": state_dict,
        "epoch": None if epoch is None else int(epoch),
        "model_tag": "B",
        "config": {
            "num_classes": 2,
            "features": FEATURES if "FEATURES" in globals() else None,
            "blocks_per_stage": BLOCKS_PER_STAGE if "BLOCKS_PER_STAGE" in globals() else None,
            "strides": STRIDES if "STRIDES" in globals() else None,
            "roi": TRAIN_PATCH if "TRAIN_PATCH" in globals() else None,
            "ignore_label": 255,
            "fg_is_label_1": True,
            "label_2_is_ignore": True,
        },
        "extra": extra if isinstance(extra, dict) else {},
        "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    tmp = path_out + ".tmp"
    torch.save(ckpt, tmp)
    os.replace(tmp, path_out)

# Prefer best -> final -> latest pt
cands = []
best = os.path.join(CKPT_DIR, "model_b_best.pt")
final = os.path.join(CKPT_DIR, "model_b.pt")
if os.path.exists(best): cands.append(best)
if os.path.exists(final): cands.append(final)
cands += sorted(glob.glob(os.path.join(CKPT_DIR, "model_b_epoch*.pt")))

if not cands:
    any_pt = sorted(glob.glob(os.path.join(CKPT_DIR, "*.pt")))
    cands += any_pt

assert cands, f"[FATAL] No checkpoints found in {CKPT_DIR}"

src = cands[0]
sd, raw = _load_state_dict_any(src)
_wrap_and_save(os.path.join(CKPT_DIR, "model_b.pt"), sd, epoch=None, extra={"canonical_src": src, "raw": raw})

meta = {
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "canonical_src": src,
    "model_b_pt_exists": True,
}
with open(os.path.join(CKPT_DIR, "model_b_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(f"[ARTIFACT] model_b.pt written (wrapped) from: {src}")
print("[ARTIFACT] model_b_meta.json written")